# Task 4: Build A Speech-to-Reasoning Pipeline With Whisper & Quantized LLM

This notebook implements an end-to-end **Speech-to-Reasoning** pipeline:

1. **Whisper ASR** — OpenAI's Whisper transcribes audio input to text
2. **Quantized Reasoning LLM** — An Unsloth dynamic 4-bit model performs logical reasoning on the transcribed text

Pipeline: `Audio → Whisper (ASR) → Transcribed Text → Quantized LLM → Reasoned Response`

Key considerations:
- Efficient GPU memory management (Whisper + LLM sharing a single T4 GPU)
- Proper encoding and batching
- Sample audio demo for end-to-end validation

## 1. Install Required Libraries

In [ ]:
%%capture
%pip install unsloth
%pip install openai-whisper
%pip install gTTS soundfile librosa

## 2. Import Libraries & Check GPU

In [ ]:
import torch
import gc
import whisper
import numpy as np
import soundfile as sf
import librosa
from gtts import gTTS
from IPython.display import Audio, display
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime > Change runtime type > GPU (T4).")

## 3. Create Sample Audio Files

We generate sample audio queries using Google Text-to-Speech (gTTS) to demonstrate the pipeline end-to-end. In production, these would be real audio recordings.

In [ ]:
import os
os.makedirs("audio_samples", exist_ok=True)

# Sample audio queries — medical reasoning questions
sample_queries = {
    "query_1.mp3": "A 55 year old male presents with sudden onset crushing chest pain radiating to his left arm, diaphoresis, and shortness of breath. His ECG shows ST elevation in leads V1 through V4. What is the most likely diagnosis, and what is the immediate management?",
    "query_2.mp3": "A 30 year old woman returns from a 12 hour international flight and presents with unilateral leg swelling, redness, and pain in her right calf. She also complains of sudden onset shortness of breath. What conditions should be suspected and how should she be evaluated?",
    "query_3.mp3": "What are the key differences between Type 1 and Type 2 diabetes, and what are the first line treatments for each?",
}

for filename, text in sample_queries.items():
    filepath = os.path.join("audio_samples", filename)
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save(filepath)
    print(f"Created: {filepath}")

print(f"\nGenerated {len(sample_queries)} audio samples")

In [ ]:
# Preview the first audio sample
print("Preview of audio query 1:")
display(Audio("audio_samples/query_1.mp3"))

## 4. Load Whisper ASR Model

We load OpenAI's Whisper model for speech-to-text transcription. Using the `base` model to balance accuracy and VRAM usage — it only needs ~1 GB of VRAM, leaving the rest for the reasoning LLM.

In [ ]:
# Load Whisper model (base model: good accuracy, ~1 GB VRAM)
whisper_model = whisper.load_model("base", device="cuda")

print(f"Whisper model loaded on: {next(whisper_model.parameters()).device}")
print(f"VRAM after Whisper: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Transcribe Audio with Whisper

Test the Whisper pipeline on our sample audio files.

In [ ]:
def transcribe_audio(audio_path, whisper_model):
    """Transcribe audio file to text using Whisper.

    Handles encoding by loading audio at Whisper's expected 16kHz sample rate,
    and processes in 30-second batches internally.
    """
    # Load and preprocess audio to 16kHz mono (Whisper's expected format)
    audio = whisper.load_audio(audio_path)
    duration = len(audio) / 16000  # 16kHz sample rate

    print(f"  Audio duration: {duration:.1f}s | Samples: {len(audio):,}")

    # Transcribe with fp16 for efficiency on GPU
    result = whisper_model.transcribe(
        audio,
        language="en",
        fp16=torch.cuda.is_available(),
    )

    return result["text"].strip()


# Transcribe all samples
transcriptions = {}
for filename in sample_queries:
    filepath = os.path.join("audio_samples", filename)
    print(f"\nTranscribing: {filename}")
    text = transcribe_audio(filepath, whisper_model)
    transcriptions[filename] = text
    print(f"  Transcription: {text}")

## 6. Free Whisper VRAM & Load Quantized Reasoning LLM

To efficiently manage GPU memory on a T4 (15 GB VRAM), we unload Whisper before loading the reasoning LLM. This sequential approach maximizes available memory for each model.

In [ ]:
# Free Whisper VRAM
del whisper_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after freeing Whisper: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Load Unsloth dynamic 4-bit quantized reasoning model
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect dtype
    load_in_4bit=True,    # Dynamic 4-bit quantization
)

# Switch to inference mode
FastLanguageModel.for_inference(model)

print(f"\nReasoning LLM loaded: {MODEL_NAME}")
print(f"VRAM after LLM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 7. Build the Reasoning Pipeline

The reasoning component takes the transcribed text and generates a detailed, logical response using chain-of-thought reasoning.

In [ ]:
def reason_from_text(transcribed_text, max_new_tokens=1024, temperature=0.7):
    """Generate a reasoned response from transcribed text using the quantized LLM."""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical expert with advanced knowledge in clinical reasoning, "
                "diagnostics, and treatment planning. When answering questions, think "
                "step-by-step through the clinical scenario. Consider the differential "
                "diagnosis, relevant investigations, and evidence-based management. "
                "Be thorough yet concise."
            )
        },
        {
            "role": "user",
            "content": transcribed_text
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    print(f"  Input tokens: {inputs.shape[1]}")

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    outputs = model.generate(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        use_cache=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

## 8. End-to-End Pipeline Function

Combines both stages into a single callable pipeline.

In [ ]:
def speech_to_reasoning(audio_path=None, transcription=None, max_new_tokens=1024):
    """
    Full speech-to-reasoning pipeline.

    Args:
        audio_path: Path to audio file (if Whisper is loaded)
        transcription: Pre-computed transcription (if Whisper already ran)
        max_new_tokens: Max tokens for LLM generation
    """
    print("=" * 70)
    print("SPEECH-TO-REASONING PIPELINE")
    print("=" * 70)

    # Stage 1: Transcription
    if transcription:
        text = transcription
    elif audio_path:
        print(f"\n[Stage 1] Transcribing: {audio_path}")
        text = transcribe_audio(audio_path, whisper_model)
    else:
        raise ValueError("Provide either audio_path or transcription")

    print(f"\n[Transcription]")
    print(f"  {text}")

    # Stage 2: Reasoning
    print(f"\n[Stage 2] Generating reasoned response...")
    print(f"  VRAM in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print()

    response = reason_from_text(text, max_new_tokens=max_new_tokens)

    print("\n" + "=" * 70)
    return {"transcription": text, "response": response}

## 9. Demo: Run the Full Pipeline

We run the end-to-end pipeline using the transcriptions from Whisper (Stage 1 already completed above) and the quantized reasoning LLM (Stage 2).

In [ ]:
# Demo 1: STEMI heart attack scenario
print("\n" + "#" * 70)
print("DEMO 1: Acute Chest Pain Scenario")
print("#" * 70)
result_1 = speech_to_reasoning(transcription=transcriptions["query_1.mp3"])

In [ ]:
# Demo 2: DVT / PE scenario after long flight
print("\n" + "#" * 70)
print("DEMO 2: Post-Travel DVT/PE Scenario")
print("#" * 70)
result_2 = speech_to_reasoning(transcription=transcriptions["query_2.mp3"])

In [ ]:
# Demo 3: Diabetes comparison
print("\n" + "#" * 70)
print("DEMO 3: Diabetes Type 1 vs Type 2")
print("#" * 70)
result_3 = speech_to_reasoning(transcription=transcriptions["query_3.mp3"])

## 10. Full End-to-End Demo (Whisper → LLM in One Pass)

Here we demonstrate the complete pipeline from raw audio to reasoned output. We reload Whisper, transcribe a new query, unload Whisper, load the LLM, and generate the response — all in a single flow.

In [ ]:
# Create a new audio sample for the full end-to-end test
e2e_query = "A patient presents with a severe headache described as the worst headache of their life, along with neck stiffness and photophobia. What is the most likely diagnosis and what immediate steps should be taken?"
e2e_audio_path = "audio_samples/e2e_query.mp3"
tts = gTTS(text=e2e_query, lang='en', slow=False)
tts.save(e2e_audio_path)
print(f"Created end-to-end test audio: {e2e_audio_path}")
display(Audio(e2e_audio_path))

In [ ]:
print("=" * 70)
print("FULL END-TO-END PIPELINE")
print("=" * 70)

# Stage 1: Unload LLM, Load Whisper, Transcribe
print("\n[Step 1] Freeing LLM VRAM...")
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"  VRAM after freeing LLM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\n[Step 2] Loading Whisper for transcription...")
whisper_model = whisper.load_model("base", device="cuda")
print(f"  VRAM after Whisper: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\n[Step 3] Transcribing audio...")
e2e_transcription = transcribe_audio(e2e_audio_path, whisper_model)
print(f"\n  Transcription: {e2e_transcription}")

# Stage 2: Unload Whisper, Load LLM, Reason
print("\n[Step 4] Freeing Whisper, loading reasoning LLM...")
del whisper_model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f"  VRAM after LLM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\n[Step 5] Generating reasoned response...")
print()
e2e_response = reason_from_text(e2e_transcription)

print("\n" + "=" * 70)
print("END-TO-END PIPELINE COMPLETE")
print("=" * 70)

## 11. GPU Memory Summary

Final report on GPU memory usage throughout the pipeline.

In [ ]:
print("=" * 55)
print("SPEECH-TO-REASONING PIPELINE — MEMORY REPORT")
print("=" * 55)
print(f"Whisper Model:    base (~1 GB VRAM)")
print(f"Reasoning Model:  {MODEL_NAME}")
print(f"Quantization:     Dynamic 4-bit (bitsandbytes NF4)")
print(f"Max seq length:   {MAX_SEQ_LENGTH}")
print(f"")
print(f"Strategy: Sequential model loading")
print(f"  1. Load Whisper → Transcribe all audio → Free Whisper")
print(f"  2. Load LLM → Generate reasoning for all queries → Done")
print(f"  This keeps peak VRAM usage under ~6 GB, well within")
print(f"  Colab T4's 15 GB limit.")
print(f"")
print(f"Audio samples processed: {len(transcriptions) + 1}")
print(f"Current VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Current VRAM reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")